# Freespan and VIV screening analysis

## Summary

This walkthrough recomputes the repository demo's current-only freespan screening
matrix, explains one beam-frequency calculation, and compares cross-flow onset
spans across pipe sizes and seabed gaps. The figures and printed results come
from the code cells below. Agreement with the demo establishes reproducibility;
standards compliance and fatigue life are not established.

## Revision history

| Revision | Date | What changed and why |
|---|---|---|
| 0 | 2026-10-14 | Initial computed walkthrough of the existing screening model. |

Table 1. Notebook revision history.

## 1. Methodology and computational basis

The existing demo identifies DNV-RP-F105, *Free Spanning Pipelines*, 2017 edition.
The licensed text and its clause titles have not been independently checked here;
no clause-level conformance claim is made. No standard table or coefficient is
transcribed into this notebook. Model constants are imported from the repository.

The calculation path is geometry → bending stiffness and effective mass → first
natural frequency → reduced velocity → the demo's in-line/cross-flow status.
The reported allowable span is specifically the model's cross-flow onset span,
not an overall design allowable span. In-line vibration may occur below that limit.

Sources are `examples/demos/gtm/demo_01_dnv_freespan_viv.py`,
`examples/demos/gtm/sweep_config.py`, `examples/demos/gtm/inputs/demo_01_freespan.yml`,
and the two catalogs resolved by that input. The source digests are computed below.
The code-faithful limitations are documented in
`examples/demos/gtm/docs/demo_01_analysis.md`.

The geometry, contents and loading are demonstration assumptions, based on
previous experience, as encoded in the repository; they are not site measurements.
`digitalmodel` owns this derived computational example. The input digests identify
the exact source bytes; a new run is required after a source change.

### Execution

Run from the repository root or from this notebook directory with the project
environment. Saving with a Python kernel can use nbclient 0.10.2, nbformat 5.10.4
and ipykernel 6.29.5; those version tags are published in their respective Jupyter
and IPython GitHub repositories. These are execution tools, not project dependencies.
Plotly figures use notebook MIME data, with no external script, stylesheet or font
output. Interaction requires a notebook viewer supporting the Plotly MIME type.
The numerical tables remain readable without that viewer.


In [1]:
import hashlib
import math
import sys
from pathlib import Path

import plotly.graph_objects as go

root = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "examples/demos/gtm/demo_01_dnv_freespan_viv.py").is_file()
)
demo_dir = root / "examples/demos/gtm"
sys.path.insert(0, str(demo_dir))
import demo_01_dnv_freespan_viv as demo
from sweep_config import load_sweep_config

input_path = demo.DEFAULT_SWEEP_CONFIG_PATH
config = load_sweep_config(input_path)
pipe_path = demo._resolve_catalog_path(input_path.parent, config.pipelines_catalog)
jumper_path = demo._resolve_catalog_path(input_path.parent, config.jumpers_catalog)
_, pipes = demo.load_pipe_catalog(pipe_path)
_, jumper = demo.load_jumper_catalog(jumper_path)
for source in (Path(demo.__file__), demo_dir / "sweep_config.py",
               input_path, pipe_path, jumper_path):
    print(f"{source.relative_to(root)} | SHA-256 {hashlib.sha256(source.read_bytes()).hexdigest()}")
results, frame = demo.run_parametric_sweep(pipes, jumper, config)
print(f"Computed cases: {len(results)}")
print("Status | Cases")
for status, count in frame.groupby("status").size().items():
    print(f"{status} | {count}")


examples/demos/gtm/demo_01_dnv_freespan_viv.py | SHA-256 0cf1b85a187061febe53f23867b050d62e0c3cd9a76cc118b3667f3776b9c5f9
examples/demos/gtm/sweep_config.py | SHA-256 197fc7b1e9d6223f723172d0fbbb18b4e8f8f2e39ce355d998192ec503569386
examples/demos/gtm/inputs/demo_01_freespan.yml | SHA-256 5ba04a6db07a7e93a7cb71347ef249ff5534b5cb68c4084da8937cfcdedf7e33
examples/demos/gtm/data/pipelines.json | SHA-256 da5f620d7bf0ff5d870b10693a040fa378ff6b3b1f7aa75eae12b23025e7550f
examples/demos/gtm/data/rigid_jumpers.json | SHA-256 36fc1f0dd4a85637a4d75703b4c8ae51278e71061da205b1b0d67237b1c8153d
  Pipeline cases: 480
    Completed 480 pipeline cases
  Jumper cases: 200
    Completed 200 jumper cases
Computed cases: 680
Status | Cases
FAIL_CF | 444
FAIL_LOCKIN | 79
INLINE_ONLY | 89
PASS | 68


## 2. Natural frequency walkthrough

The first pipeline case from the configured matrix is used. Its wall thickness
is the catalog's thinnest option and its support condition is the input's boundary
label. The demo computes `I = pi (OD^4 - ID^4) / 64` and sums steel, coating,
contents and hydrodynamic added mass. Then
`f_n = C_n / (2 pi) sqrt(E I / (m_eff L^4))`.
The boundary coefficient is reused exactly as implemented, without independent
qualification of its normalization. No axial force, soil stiffness or multiple
span interaction is included.


In [2]:
case = results[0]
pipe = demo.get_pipe_by_size(pipes, case["nominal_size"])
density = config.pipelines.content_density_kg_m3[0]
boundary = config.pipelines.c_n_values[0]
coating = pipe["anti_corrosion_coating"]["thickness_m"]
masses = demo.calc_mass_per_meter(
    case["od_m"], case["wt_m"], content_density=density,
    coating_thickness_m=coating,
)
inertia = demo.calc_second_moment_of_area(case["od_m"], case["wt_m"])
frequency_from_parts = boundary / (2 * math.pi) * math.sqrt(
    demo.STEEL_YOUNGS_MODULUS * inertia / (masses["m_eff"] * case["span_m"] ** 4)
)
print(f"Size: {case['nominal_size']}")
print(f"Outside diameter (m): {case['od_m']:.3f}")
print(f"Wall thickness (m): {case['wt_m']:.3f}")
print(f"Coating thickness (m): {coating:.3f}")
print(f"Span (m): {case['span_m']:.1f}")
print(f"Support condition: {config.pipelines.boundary_condition_labels[0]}")
print(f"Content density (kg/m3): {density:.1f}")
for name, mass in masses.items():
    print(f"{name} (kg/m): {mass:.1f}")
print(f"Natural frequency (Hz): {case['f_n_hz']:.1f}")
print(f"Frequency from stiffness and mass (Hz): {frequency_from_parts:.1f}")


Size: 8in
Outside diameter (m): 0.219
Wall thickness (m): 0.008
Coating thickness (m): 0.003
Span (m): 10.0
Support condition: pinned
Content density (kg/m3): 800.0
m_steel (kg/m): 42.6
m_coating (kg/m): 2.0
m_content (kg/m): 25.8
m_added (kg/m): 38.6
m_eff (kg/m): 109.1
Natural frequency (Hz): 1.4
Frequency from stiffness and mass (Hz): 1.4


## 3. Reduced velocity and VIV onset

`V_R = current / (f_n OD)` is evaluated by the demo. Its onset thresholds and gap
correction are imported, rather than restated as values from the recommended
practice. The status is the demo's simplified screening classification. A `PASS`
means reduced velocity is below this model's in-line onset; it does not establish
fatigue acceptance. The comparison is made on the unrounded module calculation;
display rounding is not used to assign status.


In [3]:
frequency = demo.calc_natural_frequency(
    case["od_m"], case["wt_m"], case["span_m"],
    content_density=density, coating_thickness_m=coating, c_n=boundary,
)
velocity = demo.calc_reduced_velocity(case["v_current_ms"], frequency, case["od_m"])
status, amplitude = demo.screen_viv(velocity, case["e_over_d"])
print(f"Current (m/s): {case['v_current_ms']:.1f}")
print(f"Gap ratio e/D (-): {case['e_over_d']:.1f}")
print(f"Reduced velocity (-): {case['v_r']:.1f}")
print(f"Screening status: {status}")
print(f"Response amplitude A/D (-): {amplitude:.1f}")


Current (m/s): 0.2
Gap ratio e/D (-): 0.5
Reduced velocity (-): 0.7
Screening status: PASS
Response amplitude A/D (-): 0.0


## 4. Gap ratio and cross-flow onset span

For each configured pipeline size, current and gap ratio, the demo's status-bearing
span search is called. Only a bracketed cross-flow onset is displayed as a finite
span. No-current and no-onset-in-range results are indeterminate span limits, not
a finite allowable span. The configured cases below all bracket onset.

The demo changes the onset criterion with gap ratio, while holding its added-mass
model fixed. A larger computed span near the seabed therefore reflects this model's
onset correction alone; it does not justify seabed clearance or fatigue acceptance.


In [4]:
span_rows = []
print("Size | Current (m/s) | Gap ratio (-) | CF onset span (m)")
for size in config.pipelines.sizes:
    selected = demo.get_pipe_by_size(pipes, size)
    wall = demo.get_thinnest_wt(selected)["wt_m"]
    for gap in config.pipelines.gap_ratios:
        for current in config.pipelines.current_velocities_ms:
            limit = demo.calc_max_allowable_span_status(
                selected["od_m"], wall, current, e_over_d=gap,
                content_density=density,
                coating_thickness_m=selected["anti_corrosion_coating"]["thickness_m"],
                c_n=boundary,
            )
            if limit["status"] != demo.MAX_SPAN_STATUS_OK:
                raise ValueError("A plotted case has no bracketed cross-flow onset")
            span_rows.append(dict(size=size, current=current, gap=gap,
                                  span=limit["span_m"], status=limit["status"]))
            print(f"{size} | {current:.1f} | {gap:.1f} | {limit['span_m']:.1f}")
print("Table 2. Computed cross-flow onset spans; not full design allowable spans.")


Size | Current (m/s) | Gap ratio (-) | CF onset span (m)
8in | 0.2 | 0.5 | 23.6
8in | 0.4 | 0.5 | 16.7
8in | 0.6 | 0.5 | 13.6
8in | 0.8 | 0.5 | 11.8
8in | 1.0 | 0.5 | 10.6
8in | 0.2 | 1.0 | 21.1
8in | 0.4 | 1.0 | 14.9
8in | 0.6 | 1.0 | 12.2
8in | 0.8 | 1.0 | 10.6
8in | 1.0 | 1.0 | 9.4
8in | 0.2 | 2.0 | 21.1
8in | 0.4 | 2.0 | 14.9
8in | 0.6 | 2.0 | 12.2
8in | 0.8 | 2.0 | 10.6
8in | 1.0 | 2.0 | 9.4
8in | 0.2 | 5.0 | 21.1
8in | 0.4 | 5.0 | 14.9
8in | 0.6 | 5.0 | 12.2
8in | 0.8 | 5.0 | 10.6
8in | 1.0 | 5.0 | 9.4
12in | 0.2 | 0.5 | 33.7
12in | 0.4 | 0.5 | 23.8
12in | 0.6 | 0.5 | 19.5
12in | 0.8 | 0.5 | 16.9
12in | 1.0 | 0.5 | 15.1
12in | 0.2 | 1.0 | 30.1
12in | 0.4 | 1.0 | 21.3
12in | 0.6 | 1.0 | 17.4
12in | 0.8 | 1.0 | 15.1
12in | 1.0 | 1.0 | 13.5
12in | 0.2 | 2.0 | 30.1
12in | 0.4 | 2.0 | 21.3
12in | 0.6 | 2.0 | 17.4
12in | 0.8 | 2.0 | 15.1
12in | 1.0 | 2.0 | 13.5
12in | 0.2 | 5.0 | 30.1
12in | 0.4 | 5.0 | 21.3
12in | 0.6 | 5.0 | 17.4
12in | 0.8 | 5.0 | 15.1
12in | 1.0 | 5.0 | 13.5
16in |

## 5. Interactive cross-flow onset span maps

Hover for the computed span at a size/current/gap combination. Axes are the input
matrix, and every heatmap value comes from the preceding module calls. Displayed
spans are rounded; the model search uses unrounded values.


In [5]:
import json

for size in config.pipelines.sizes:
    z = [
        [round(row["span"], 1) for row in span_rows
         if row["size"] == size and row["gap"] == gap]
        for gap in config.pipelines.gap_ratios
    ]
    figure = go.Figure(go.Heatmap(
        x=config.pipelines.current_velocities_ms,
        y=config.pipelines.gap_ratios,
        z=z, colorbar=dict(title="Span (m)"),
        hovertemplate="Current %{x:.1f} m/s<br>Gap %{y:.1f}<br>CF onset %{z:.1f} m<extra></extra>",
    ))
    figure.update_layout(
        template=None, title=f"{size}: model cross-flow onset span",
        xaxis_title="Current (m/s)", yaxis_title="Gap ratio e/D (-)",
    )
    display({"application/vnd.plotly.v1+json": json.loads(figure.to_json())}, raw=True)
    print(f"Figure: {size} cross-flow onset span map (m), from computed rows.")


Figure: 8in cross-flow onset span map (m), from computed rows.


Figure: 12in cross-flow onset span map (m), from computed rows.


Figure: 16in cross-flow onset span map (m), from computed rows.


## 6. Pipe-size comparison and way forward

The comparison holds current, gap, contents density and support label fixed at
the first configured values. Each size retains its catalog geometry and thinnest
wall, so this is a catalog comparison rather than an isolated diameter experiment.
The difference is relative to the first configured size; the values are calculated
before display rounding. An increase is a screening-model span increase only.


In [6]:
comparison = [row for row in span_rows
              if row["current"] == config.pipelines.current_velocities_ms[0]
              and row["gap"] == config.pipelines.gap_ratios[0]]
reference_span = comparison[0]["span"]
print(f"Current (m/s): {comparison[0]['current']:.1f}; gap ratio (-): {comparison[0]['gap']:.1f}")
print("Size | Wall thickness (m) | CF onset span (m) | Difference (m) | Difference (%)")
for row in comparison:
    selected = demo.get_pipe_by_size(pipes, row["size"])
    wall = demo.get_thinnest_wt(selected)["wt_m"]
    difference = row["span"] - reference_span
    percentage = difference / reference_span * 100
    print(f"{row['size']} | {wall:.3f} | {row['span']:.1f} | {difference:.1f} | {percentage:.1f}")
print("Table 3. Pipe-size comparison against the first configured size.")


Current (m/s): 0.2; gap ratio (-): 0.5
Size | Wall thickness (m) | CF onset span (m) | Difference (m) | Difference (%)
8in | 0.008 | 23.6 | 0.0 | 0.0
12in | 0.009 | 33.7 | 10.1 | 42.7
16in | 0.009 | 40.8 | 17.1 | 72.6
Table 3. Pipe-size comparison against the first configured size.


A size change may improve the modeled cross-flow onset span, but an upsize decision
also requires pressure containment, installation, weight and cost checks. Before
engineering use, qualify the boundary coefficient and gap correction against the
licensed edition, confirm actual geometry and environmental data, include axial
force and soil support, and perform in-line/cross-flow fatigue assessment with
appropriate stress concentration factors and exposure distributions. This notebook
establishes repeatable demo calculations only; those design checks remain open.
